<a href="https://colab.research.google.com/github/jagadeesh-usd/composer-classification/blob/jag-dev/notebooks/01_Data_Preprocessing_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Preprocessing for Composer Classification

## 1. Introduction
This notebook handles the entire data preprocessing pipeline for our composer classification project. The primary goal is to convert a dataset of raw MIDI files from four composers (Bach, Beethoven, Chopin, and Mozart) into a clean, structured format suitable for training deep learning models.

The key steps in this process are:
1.  **Configuration:** Define paths, model parameters, and composers.
2.  **Feature Extraction:** Parse MIDI files to extract musical features like notes, chords, and tempo.
3.  **Data Splitting:** Strategically split the data into training, validation, and test sets to prevent data leakage.
4.  **Data Augmentation:** Apply pitch transposition to the training data to increase its diversity and help the model generalize better.
5.  **Sequence Generation:** Convert the long streams of notes from each musical piece into fixed-length sequences.
6.  **Vocabulary Creation:** Build a numerical vocabulary to map musical elements to integers.
7.  **Saving Data:** Save the final processed datasets and preprocessing objects for use in the model training notebook.

In [2]:
# 1. Import Required Libraries
import os
import time
import glob
import pickle
from multiprocessing import Pool, cpu_count
from collections import Counter

from music21 import converter, instrument, note, chord

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 2. Configuration

This section defines the core hyperparameters and configuration settings that control the entire preprocessing pipeline. Centralizing these parameters makes it easy to experiment and modify the process.

- **`DATA_PATH`**: The root directory where the composer-specific MIDI files are stored.
- **`COMPOSERS`**: A list of the composers we will include in our dataset. For this project, we are focusing on four key figures from the classical and romantic eras.
- **`SEQUENCE_LENGTH`**: This defines the fixed length of each input sequence for our model. A sequence of 100 notes provides enough context for the model to learn meaningful patterns without being computationally prohibitive.
- **`VOCAB_SIZE`**: The total number of unique musical elements (notes and chords) to include in our vocabulary. This helps manage the model's complexity by filtering out extremely rare elements.
- **`SEMITONES_TO_TRANSPOSE`**: A list of intervals (in semitones) used for data augmentation. Transposing pieces up and down in pitch helps the model learn melodic and harmonic patterns that are independent of the original key.

In [3]:
# 2. Configuration
DATA_PATH = "/content/drive/My Drive/composer_project/data/midi_files/train"
# COMPOSERS = ["Bach", "byrd", "Chopin", "Handel", "Mozart", "schumann"]
COMPOSERS = ["Bach", "Beethoven", "Chopin",  "Mozart"]

# COMPOSERS = ["Bach",  "Chopin",  "Mozart"]

# COMPOSERS = ["Bach", "byrd", "Chopin",  "Mozart"]

SEQUENCE_LENGTH = 100 # The length of a single training sequence
VOCAB_SIZE = 2000 # The number of unique notes/chords to keep
# Augmentation setting
SEMITONES_TO_TRANSPOSE = [2, -2] # Transpose up 2 and down 2 semitones

## 3. Feature Extraction and Augmentation Functions

Feature extraction is the process of converting raw data (MIDI files) into a format our model can understand. We define two key functions for this purpose:

1.  **`extract_features_from_file(file_path)`**: This function takes the path to a single MIDI file and uses the `music21` library to parse it. It iterates through the musical elements and extracts two primary features:
    - **Notes and Chords**: Individual notes are stored by their pitch name (e.g., 'C#4'), while chords are stored as a dot-separated string of their constituent note pitches in a normalized form (e.g., '0.4.7' for a C major chord).
    - **Average Tempo**: The function calculates the average tempo of the piece in beats per minute (BPM), which will be used as an auxiliary numerical feature.

2.  **`transpose_sequence(sequence, semitones)`**: This function handles our data augmentation. It takes a sequence of notes and chords and transposes each element up or down by a specified number of semitones, creating a new, musically-related training example.

In [4]:
#3. Feature Extraction and Augmentation Functions

def transpose_sequence(sequence, semitones):
    """Transposes a sequence of notes and chords by a number of semitones."""
    transposed_seq = []
    for element in sequence:
        try:
            # Check if the element is a valid note or chord
            if '.' in element:
                # Check if it's a valid chord
                chord_parts = element.split('.')
                valid_chord = True
                for part in chord_parts:
                    if not (len(part) >= 2 and part[1:].isdigit() and part[0].isalpha()):
                        valid_chord = False
                        break
                if valid_chord:
                    c = chord.Chord(chord_parts)
                    c.transpose(semitones, inPlace=True)
                    transposed_seq.append('.'.join(str(n) for n in c.normalOrder))
                else:
                    transposed_seq.append(element)  # Keep original element if it's not a valid chord
            else:
                # Check if it's a valid note
                if len(element) >= 2 and element[1:].isdigit() and element[0].isalpha():
                    n = note.Note(element)
                    n.transpose(semitones, inPlace=True)
                    transposed_seq.append(str(n.pitch))
                else:
                    transposed_seq.append(element)  # Keep original element if it's not a valid note
        except Exception as e:
            print(f"Error transposing element {element}: {e}")
            transposed_seq.append(element)  # Keep original element if transposition fails
    return transposed_seq


def extract_features_from_file(file_path):
    """Extracts notes, chords, and average tempo from a single MIDI file."""
    notes = []
    avg_tempo = 120.0 # Default tempo
    try:
        midi = converter.parse(file_path)

        # Extract tempo
        tempo_marks = midi.flatten().getElementsByClass('TempoIndication')
        if tempo_marks:
            avg_tempo = np.mean([t.number for t in tempo_marks if hasattr(t, 'number')])

        # Extract notes and chords
        parts = instrument.partitionByInstrument(midi)
        notes_to_parse = parts.parts[0].recurse() if parts else midi.flatten().notes
        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))
    except Exception as e:
        print(f"  Error processing {os.path.basename(file_path)}: {e}")

    return {'notes': notes, 'avg_tempo': avg_tempo}

## 4. Main Data Processing and Splitting

This section contains the main logic for building our datasets. It follows a careful, multi-step process to ensure data integrity and prevent data leakage.

### 4.1 File-Level Data Splitting
A robust data splitting strategy is crucial for a valid model evaluation. Instead of splitting the final sequences, we split the *files* themselves. This ensures that sequences from the same musical piece cannot appear in both the training and testing sets, which would lead to an artificially inflated performance metric.

The process is as follows:
1.  **Initial Train/Test Split (80/20):** We first perform a stratified split on the entire file list. Stratifying by composer ensures that the proportion of pieces from each composer is the same in both the training and test sets.
2.  **Secondary Train/Validation Split (80/20 of the training set):** We then take the initial training set of files and perform a second stratified split to create our final training and validation sets. This provides an unbiased validation set for monitoring model performance during training.

### 4.2 Sequence Generation and Augmentation
The `process_files` function orchestrates the conversion of files into sequences. It uses a **sliding window** approach, moving across the stream of notes extracted from a piece.

- **`SEQUENCE_LENGTH`**: Defines the size of the window (100 notes).
- **`STRIDE`**: Defines how many notes the window moves forward at each step. We use a stride of 10 to generate a diverse but manageable number of sequences from each piece. A smaller stride would result in a much larger, highly redundant dataset.

Crucially, data augmentation (pitch transposition) is applied **only to the training set** (`augment=True`). This enriches the training data without contaminating the validation and test sets with artificial data, ensuring they remain a true measure of the model's generalization ability.

In [5]:
# 4. Main Processing Logic

# Create a DataFrame of all file paths and their composers
all_files = []
for composer in COMPOSERS:
    composer_path = os.path.join(DATA_PATH, composer.lower())  # ensure this matches your folder casing
    if os.path.isdir(composer_path):
        files = glob.glob(os.path.join(composer_path, '*.mid*'))
        for f in files:
            all_files.append({'path': f, 'composer': composer})
files_df = pd.DataFrame(all_files)

# 4.1 File-level Train/Test split (stratified by composer)
train_files_df, test_files_df = train_test_split(
    files_df, test_size=0.20, random_state=42, stratify=files_df['composer']
)
print(f"Files → train: {len(train_files_df)} | test: {len(test_files_df)}")

# 4.2 File-level Train/Val split from *train_files_df* (by composer)
val_ratio = 0.20
train_core_parts, val_parts = [], []
for comp, g in train_files_df.groupby('composer'):
    t, v = train_test_split(g, test_size=val_ratio, random_state=42)  # group-wise, so composer stays balanced
    train_core_parts.append(t)
    val_parts.append(v)
train_core_df = pd.concat(train_core_parts, ignore_index=True)
val_files_df  = pd.concat(val_parts,       ignore_index=True)
print(f"Files → train_core: {len(train_core_df)} | val: {len(val_files_df)} | test: {len(test_files_df)}")

def process_files(df, augment=False, stride=1):
    """
    df: DataFrame with columns ['path','composer']
    augment: apply pitch-shift augmentation (TRAIN ONLY)
    stride: step size for sliding windows (>=1)
    """
    sequences, labels, tempos, all_notes = [], [], [], []
    for _, row in df.iterrows():
        features = extract_features_from_file(row['path'])
        notes = features['notes']
        if len(notes) > SEQUENCE_LENGTH:
            # original windows (use stride to cut dataset size)
            orig = [notes[i:i+SEQUENCE_LENGTH] for i in range(0, len(notes)-SEQUENCE_LENGTH, stride)]
            sequences.extend(orig)
            labels.extend([row['composer']]*len(orig))
            tempos.extend([features['avg_tempo']]*len(orig))
            all_notes.extend(notes)

            # augmentation (train only)
            if augment:
                for s in SEMITONES_TO_TRANSPOSE:
                    tnotes = transpose_sequence(notes, s)
                    aug = [tnotes[i:i+SEQUENCE_LENGTH] for i in range(0, len(tnotes)-SEQUENCE_LENGTH, stride)]
                    sequences.extend(aug)
                    labels.extend([row['composer']]*len(aug))
                    tempos.extend([features['avg_tempo']]*len(aug))
    return sequences, labels, tempos, all_notes

# Use stride=2 or 4 to speed up training substantially
STRIDE = 10

# Build datasets (augmentation TRAIN ONLY)
train_sequences, train_labels, train_tempos, train_all_notes = process_files(train_core_df, augment=True,  stride=STRIDE)
val_sequences,   val_labels,   val_tempos,   _               = process_files(val_files_df,   augment=False, stride=STRIDE)
test_sequences,  test_labels,  test_tempos,  _               = process_files(test_files_df,  augment=False, stride=STRIDE)

print(f"\nSequences → train: {len(train_sequences)} | val: {len(val_sequences)} | test: {len(test_sequences)}")


Files → train: 132 | test: 33
Files → train_core: 104 | val: 28 | test: 33


/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=6, channel=None, data=b'Copyright \xa9 July 1997 by'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, channel=None, data=b'\x8b\xc8\x82\xcc\x95\\\x91\xe8'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, channel=None, data=b' Anhang 6 Rond\xf2 per pianoforte'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NA


Sequences → train: 59895 | val: 7095 | test: 5292


## 5. Create Vocabulary

To prepare our note sequences for the deep learning model, we must convert them from strings (e.g., 'C#4') into integers. This is achieved by creating a vocabulary.

The process is as follows:
1.  **Count Frequencies:** We count the occurrences of every unique note and chord across the entire **training dataset**.
2.  **Select Vocabulary:** We select the most common elements up to our defined `VOCAB_SIZE`. This helps manage the complexity of the model by ignoring extremely rare notes that might not provide a generalizable signal.
3.  **Add 'UNK' Token:** We add a special 'UNK' (unknown) token to our vocabulary. This token will be used for any notes that appear in the validation or test sets but were not part of our selected vocabulary from the training set. This prevents errors and handles out-of-vocabulary elements gracefully.
4.  **Create Mapping:** We create a dictionary, `note_to_int`, that maps each unique musical element to a unique integer index.

In [6]:
# 5. Create Vocabulary
note_counts = Counter(train_all_notes)
most_common_notes = [n for n, c in note_counts.most_common(VOCAB_SIZE - 1)]
most_common_notes.append('UNK')
note_to_int = {note: i for i, note in enumerate(most_common_notes)}

## 6. Save Processed Data for Model Training

To ensure efficiency and reproducibility, we save all our processed data and metadata to disk using `pickle`. This is a crucial step that separates the time-consuming preprocessing pipeline from the model training pipeline. It allows us to experiment with different model architectures and training configurations without needing to re-process the MIDI files every time.

We are saving four main files:
-   **`train_data.pkl`**: Contains the final training sequences, labels, and tempo data.
-   **`val_data.pkl`**: Contains the final validation sequences, labels, and tempo data.
-   **`test_data.pkl`**: Contains the final test sequences, labels, and tempo data.
-   **`preprocessing_data.pkl`**: A critical file containing the `note_to_int` vocabulary mapping and other metadata like the `SEQUENCE_LENGTH` and list of `COMPOSERS`. This ensures the training notebook has all the necessary information to correctly interpret the data.

In [7]:
# 6. Save Processed Data
print("\nSaving processed data...")
SAVE_DIR = "/content/drive/My Drive/composer_project/processed_data_augmented_with_stride10_100"
os.makedirs(SAVE_DIR, exist_ok=True)

# Combine data into dictionaries for easier loading
train_data = {'sequences': train_sequences, 'labels': train_labels, 'tempos': train_tempos}
val_data   = {'sequences': val_sequences,   'labels': val_labels,   'tempos': val_tempos}
test_data  = {'sequences': test_sequences,  'labels': test_labels,  'tempos': test_tempos}

with open(os.path.join(SAVE_DIR, 'train_data.pkl'), 'wb') as f:
    pickle.dump(train_data, f)
with open(os.path.join(SAVE_DIR, 'val_data.pkl'), 'wb') as f:
    pickle.dump(val_data, f)
with open(os.path.join(SAVE_DIR, 'test_data.pkl'), 'wb') as f:
    pickle.dump(test_data, f)

preprocessing_data = {
    'note_to_int': note_to_int,
    'composers': COMPOSERS,
    'sequence_length': SEQUENCE_LENGTH,
    'vocab_size': len(note_to_int)
}
with open(os.path.join(SAVE_DIR, 'preprocessing_data.pkl'), 'wb') as f:
    pickle.dump(preprocessing_data, f)

print(f"Data saved to {SAVE_DIR}. Preprocessing complete.")


Saving processed data...
Data saved to /content/drive/My Drive/composer_project/processed_data_augmented_with_stride10_100. Preprocessing complete.
